# DPOS End-to-End Walkthrough (Business Demo)

This notebook demonstrates DPOS end-to-end from a *functional* perspective:
- Discover data products and what they provide
- Validate incoming batches (good vs bad) against contracts
- Enforce policy (publish / block / quarantine)
- Create and handle incidents with agents (healing + stewardship)
- Understand downstream impact when a product fails
- Monitor health signals (SLA-style)
- Demonstrate streaming enforcement concepts
- Use orchestration (supervisor) to summarize and recommend next actions
- Use the same capabilities from Claude Desktop via MCP

**Use-case switching (no governance changes required):**
- Set `DPOS_USECASE` (or the `USECASE` variable below) to `ecommerce` or `covid`
- Governance stays generic; only the example dataset changes


In [1]:
# Notebook bootstrap: make imports + settings work consistently
from __future__ import annotations

import os
import sys
from pathlib import Path

try:
    from dotenv import load_dotenv
except ImportError as e:
    raise RuntimeError("Missing dependency: python-dotenv. Install repo requirements first.") from e

def find_project_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(12):
        if (cur / "src").exists() and (cur / "scripts").exists() and (cur / "config").exists():
            return cur
        cur = cur.parent
    raise RuntimeError(f"Could not find project root from: {start}")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Load business configuration (Neo4j + LLM provider)
env_path = PROJECT_ROOT / ".env"
if env_path.exists():
    load_dotenv(env_path, override=False)

# Compatibility: some parts of the repo use NEO4J_USER vs NEO4J_USERNAME
if not os.getenv("NEO4J_USER") and os.getenv("NEO4J_USERNAME"):
    os.environ["NEO4J_USER"] = os.environ["NEO4J_USERNAME"]

PROJECT_ROOT

WindowsPath('H:/akash/git/CoherencePLM/version16/dpos-ecommerce')

In [2]:
# Demo readiness check (soft): show interpreter and warn if it's not the project venv
import sys
from importlib.util import find_spec

python_path = sys.executable
print("Python:", python_path)

def looks_like_venv(p: str) -> bool:
    p_low = p.lower().replace("/", "\\")
    return "\\.venv\\" in p_low or "\\venv\\" in p_low

# Don't hard-fail here (kernels vary). Just provide a helpful warning.
if not looks_like_venv(python_path):
    print("WARNING: Kernel doesn't look like a virtualenv. If you hit import errors, switch to the repo venv.")

missing = [pkg for pkg in ["dotenv", "neo4j"] if find_spec(pkg) is None]
if missing:
    raise RuntimeError(f"Missing required packages in this kernel: {missing}. Use the repo venv + install requirements.")

python_path

Python: c:\Users\HP\AppData\Local\Programs\Python\Python312\python.exe


'c:\\Users\\HP\\AppData\\Local\\Programs\\Python\\Python312\\python.exe'

## 1) Connect to the Governance Graph
DPOS stores product definitions, contracts, validation results, incidents, and agent activity in the graph.

**Success looks like:** we can connect and query the catalog.

In [3]:
from src.graph.manager import Neo4jManager

mgr = Neo4jManager()
ok = mgr.verify_connectivity()
ok

{"timestamp": "2026-01-04T21:44:25.650261+00:00", "level": "INFO", "logger": "src.graph.manager", "message": "Neo4j driver initialized", "module": "logging", "function": "_log", "line": 134, "extra": {"uri": "neo4j+s://84dce434.databases.neo4j.io", "database": "neo4j", "pool_size": 50}}


True

## 2) Load / Refresh the Demo Catalog (One-Time)
This step creates a baseline catalog: data products, contracts, ports, pipelines, policies, and sample relationships.

**Success looks like:** later queries show multiple `DataProduct` nodes and related governance objects.

If you already loaded the demo data earlier, you can skip this.

In [4]:
import subprocess

def run_script(rel_path: str) -> None:
    script = PROJECT_ROOT / rel_path
    if not script.exists():
        raise FileNotFoundError(script)
    subprocess.run([sys.executable, str(script)], cwd=str(PROJECT_ROOT), check=True)

# Optional: initialize schema + load sample catalog/data
run_script('scripts/init_graph.py')
run_script('scripts/load_all.py')

## 3) Browse What’s Available (Catalog Discovery)
This is where you’d find products like “customer profiles”, “orders”, “transactions”, or (in a public program) “daily cases” / “vaccinations”.

**Success looks like:** you can see data products and their current status.

In [5]:
counts = mgr.execute_query("""
MATCH (n)
RETURN labels(n)[0] AS label, count(*) AS count
ORDER BY count DESC
""")
counts

[{'label': 'Field', 'count': 44},
 {'label': 'Rule', 'count': 28},
 {'label': 'AgentExecution', 'count': 20},
 {'label': 'Tag', 'count': 16},
 {'label': 'ValidationReport', 'count': 14},
 {'label': 'PolicyRule', 'count': 10},
 {'label': 'OutputPort', 'count': 8},
 {'label': 'Schema', 'count': 6},
 {'label': 'Contract', 'count': 6},
 {'label': 'SLA', 'count': 6},
 {'label': 'InputPort', 'count': 6},
 {'label': 'Pipeline', 'count': 6},
 {'label': 'User', 'count': 6},
 {'label': 'DataProduct', 'count': 6},
 {'label': 'Policy', 'count': 5},
 {'label': 'Incident', 'count': 4},
 {'label': 'Metric', 'count': 4},
 {'label': 'SupervisorExecution', 'count': 4},
 {'label': 'InsightsReport', 'count': 3},
 {'label': 'Domain', 'count': 3}]

In [6]:
products = mgr.execute_query("""
MATCH (p:DataProduct)
RETURN p.id AS id, p.name AS name, p.status AS status
ORDER BY p.id
LIMIT 10
""")
products

[{'id': 'DP001', 'name': 'customer_profiles', 'status': 'active'},
 {'id': 'DP002', 'name': 'customer_transactions', 'status': 'active'},
 {'id': 'DP003', 'name': 'orders', 'status': 'active'},
 {'id': 'DP004', 'name': 'shipments', 'status': 'active'},
 {'id': 'DP005', 'name': 'inventory_levels', 'status': 'active'},
 {'id': 'DP006', 'name': 'product_catalog', 'status': 'active'}]

## 4) Business Story (Use-case Dataset): Outreach List / Customer Contacts
We’ll load a synthetic dataset (selected by `USECASE`) and treat it as a governed data product (`DP001`).

**What we want from this product:**
- Contact details are present and well-formed
- Each person is uniquely identifiable
- Segment values are valid (used for prioritization)
- Bad records are stopped before they reach downstream consumers

In [7]:
from __future__ import annotations

import csv
import json
import os
from pathlib import Path
from typing import Any, Dict, List, Optional

PRODUCT_ID = "DP001"

# Choose which dataset to demo (governance stays the same).
# Options in this repo: "ecommerce" or "covid"
USECASE = os.getenv("DPOS_USECASE", "covid")

def read_csv(path: Path, limit: Optional[int] = None) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    with path.open("r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for idx, row in enumerate(reader):
            rows.append(dict(row))
            if limit is not None and idx + 1 >= limit:
                break
    return rows

usecase_root = PROJECT_ROOT / "data" / "usecase" / USECASE

# Prefer standard names
good_path = usecase_root / "good" / "customers.csv"
bad_path = usecase_root / "bad" / "customers_bad.csv"

# Alternative COVID names
if not good_path.exists():
    good_path = usecase_root / "good" / "outreach_list.csv"
if not bad_path.exists():
    bad_path = usecase_root / "bad" / "outreach_list_bad.csv"

# Legacy fallbacks (older layouts)
if not good_path.exists():
    good_path = PROJECT_ROOT / "data" / "good" / "customers.csv"
if not bad_path.exists():
    bad_path = PROJECT_ROOT / "data" / "bad" / "customers_bad.csv"

if not good_path.exists() or not bad_path.exists():
    raise FileNotFoundError(f"Missing demo CSVs for USECASE={USECASE}: {good_path} or {bad_path}")

good_rows = read_csv(good_path, limit=50)
bad_rows = read_csv(bad_path, limit=50)

{
    "usecase": USECASE,
    "product_id": PRODUCT_ID,
    "good_path": str(good_path),
    "bad_path": str(bad_path),
    "good_rows": len(good_rows),
    "bad_rows": len(bad_rows),
    "example_good": good_rows[0] if good_rows else None,
    "example_bad": bad_rows[0] if bad_rows else None,
}

{'usecase': 'ecommerce',
 'product_id': 'DP001',
 'good_path': 'H:\\akash\\git\\CoherencePLM\\version16\\dpos-ecommerce\\data\\usecase\\ecommerce\\good\\customers.csv',
 'bad_path': 'H:\\akash\\git\\CoherencePLM\\version16\\dpos-ecommerce\\data\\usecase\\ecommerce\\bad\\customers_bad.csv',
 'good_rows': 25,
 'bad_rows': 10,
 'example_good': {'customer_id': 'CUST000001',
  'email': 'john.smith@email.com',
  'name': 'John Smith',
  'phone': '+12025551234',
  'address': '{"street":"123 Main St","city":"New York","state":"NY","zip":"10001"}',
  'segment': 'premium',
  'created_at': '2023-01-15T10:30:00Z',
  'is_active': 'true'},
 'example_bad': {'customer_id': 'INVALID001',
  'email': 'not-an-email',
  'name': 'John Smith',
  'phone': '+12025551234',
  'address': '{"street":"123 Main St"}',
  'segment': 'premium',
  'created_at': '2023-01-15T10:30:00Z',
  'is_active': 'true'}}

## 5) Contract Validation (Good vs Bad)
We validate incoming batches against the product’s contract.

**Functional outcomes:**
- “Good” batch proceeds normally
- “Bad” batch produces a clear validation result with what failed and why

In [8]:
from src.contracts.validator import ContractValidator

validator = ContractValidator(PRODUCT_ID)

report_good = validator.validate_batch(good_rows)
report_bad = validator.validate_batch(bad_rows)

def summarize_report(r: dict) -> dict:
    return {
        "product_id": r.get("product_id"),
        "result": r.get("result"),
        "action": r.get("action"),
        "report_id": r.get("report_id"),
        "valid_records": len(r.get("valid_data", [])),
        "invalid_records": len(r.get("invalid_data", [])),
    }

{
    "good": summarize_report(report_good),
    "bad": summarize_report(report_bad),
}

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownRelationshipTypeWarning} {category: UNRECOGNIZED} {title: The provided relationship type is not in the database.} {description: One of the relationship types in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing relationship type is: APPLIES_TO_PRODUCT)} {position: line: 3, column: 18, offset: 77} for query: "\n        MATCH (p:Policy {scope:'PRODUCT', is_active:true})\n              -[:APPLIES_TO_PRODUCT]->(:DataProduct {id:$id})\n        OPTIONAL MATCH (p)-[:HAS_POLICY_RULE]->(r:PolicyRule)\n        WITH p, collect(DISTINCT r) AS rules\n        RETURN {\n            id: p.id,\n            priority: coalesce(p.priority, 0),\n            scope: 'PRODUCT',\n            rules: rules\n        } AS policy\n        "


{'good': {'product_id': 'DP001',
  'result': 'passed',
  'action': 'passed',
  'report_id': 'vr_76108a0446',
  'valid_records': 25,
  'invalid_records': 0},
 'bad': {'product_id': 'DP001',
  'result': 'failed',
  'action': 'blocked',
  'report_id': 'vr_4c4463d1d1',
  'valid_records': 6,
  'invalid_records': 4}}

In [9]:
def fetch_validation_details(report_id: str) -> dict:
    rows = mgr.execute_query(
        """
        MATCH (v:ValidationReport {id:$id})
        RETURN v AS v
        """,
        {"id": report_id},
    )
    if not rows:
        return {"error": f"ValidationReport not found: {report_id}"}
    v = dict(rows[0]["v"])
    violations_raw = v.get("violations")
    try:
        violations = json.loads(violations_raw) if isinstance(violations_raw, str) else (violations_raw or [])
    except Exception:
        violations = []
    return {
        "id": v.get("id"),
        "result": v.get("result"),
        "action": v.get("action"),
        "total_records": v.get("total_records"),
        "passed_records": v.get("passed_records"),
        "failed_records": v.get("failed_records"),
        "violations_preview": violations[:8],
    }

{
    "good_details": fetch_validation_details(report_good["report_id"]),
    "bad_details": fetch_validation_details(report_bad["report_id"]),
}

{'good_details': {'id': 'vr_76108a0446',
  'result': 'passed',
  'action': 'passed',
  'total_records': 25,
  'passed_records': 25,
  'failed_records': 0,
  'violations_preview': []},
 'bad_details': {'id': 'vr_4c4463d1d1',
  'result': 'failed',
  'action': 'blocked',
  'total_records': 10,
  'passed_records': 6,
  'failed_records': 4,
  'violations_preview': [{'rule_id': 'rule_dp001_03',
    'field': 'name',
    'severity': 'error',
    'message': "Field 'name' exceeds allowed null rate (0.0%)."},
   {'rule_id': 'rule_dp001_02',
    'field': 'email',
    'severity': 'error',
    'message': "Field 'email' does not match pattern ^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-zA-Z]{2,}$.",
    'row': 0},
   {'rule_id': 'rule_dp001_04',
    'field': 'segment',
    'severity': 'error',
    'message': "Field 'segment' must be one of ['standard', 'premium', 'vip'].",
    'row': 5},
   {'rule_id': 'rule_dp001_02',
    'field': 'email',
    'severity': 'error',
    'message': "Field 'email' does not m

## 6) Enforcement (What Happens Next)
DPOS turns validation into an operational decision.

**Functional outcomes:**
- “Publish” when the batch is acceptable
- “Block” or “Quarantine” when the batch is not acceptable
- Create an incident when the product is at risk

In [10]:
from src.enforcement.engine import EnforcementEngine

engine = EnforcementEngine()

# Enforce both outcomes. If the bad batch is blocked/quarantined, an incident is created.
engine.enforce(PRODUCT_ID, report_good)
engine.enforce(PRODUCT_ID, report_bad)

# Find the most recent incident for this product
incident_rows = mgr.execute_query(
    """
    MATCH (p:DataProduct {id:$pid})-[:HAS_INCIDENT]->(i:Incident)
    RETURN i.id AS id, i.severity AS severity, i.status AS status, i.description AS description, i.timestamp AS ts
    ORDER BY i.timestamp DESC
    LIMIT 1
    """,
    {"pid": PRODUCT_ID},
)

latest_incident = dict(incident_rows[0]) if incident_rows else None
latest_incident

[ENFORCEMENT] Action: PASSED
   [OK] Publishing 25 records to topic: dpos.customer.profiles.validated
[ENFORCEMENT] Action: BLOCKED


2026-01-05 03:15:44 | HealingAgent | INFO | Loading incident INC_ecca6472


[AGENT] Healing Agent invoked for Incident INC_ecca6472
{"timestamp": "2026-01-04T21:45:44.817518+00:00", "level": "INFO", "logger": "dpos.agents.HealingAgent", "message": "Loading incident INC_ecca6472", "module": "healing_agent", "function": "load_incident", "line": 78}


2026-01-05 03:15:45 | HealingAgent | INFO | Analyzing impact for product DP001


{"timestamp": "2026-01-04T21:45:45.139580+00:00", "level": "INFO", "logger": "dpos.agents.HealingAgent", "message": "Analyzing impact for product DP001", "module": "healing_agent", "function": "analyze_impact", "line": 91}


...

2026-01-05 03:16:25 | HealingAgent | INFO | Auto-healing: activating fallback


{"timestamp": "2026-01-04T21:46:25.831714+00:00", "level": "INFO", "logger": "dpos.agents.HealingAgent", "message": "Auto-healing: activating fallback", "module": "healing_agent", "function": "auto_heal", "line": 245}


2026-01-05 03:16:26 | HealingAgent | INFO | Persisting execution for incident INC_ecca6472


{"timestamp": "2026-01-04T21:46:26.468821+00:00", "level": "INFO", "logger": "dpos.agents.HealingAgent", "message": "Persisting execution for incident INC_ecca6472", "module": "healing_agent", "function": "persist_execution", "line": 294}
[AGENT] Healing Agent finished with state:
  Status: completed
  Action: activate_fallback
  Recommendation: Fallback source activated automatically. Root cause: A data processing error or pipeline failure introduced invalid, missing, or malformed data into the customer_profiles product, causing it to violate the defined data contract for product DP001.. Confidence: medium.


{'id': 'INC_ecca6472',
 'severity': 'high',
 'status': 'mitigating',
 'description': 'Contract violated for product DP001',
 'ts': neo4j.time.DateTime(2026, 1, 4, 21, 45, 44, 372092000, tzinfo=<UTC>)}

## 7) Agent Handling (Healing + Stewardship)
When an incident is created, agents help drive resolution and governance action.

**Functional outcomes:**
- A recommended mitigation / next action
- A governance-oriented remediation checklist

In [11]:
# Agent handling: Healing (ops) + Stewardship (governance)
# Note: if you updated code while the notebook kernel is running, reload modules first.
import importlib

import src.agents.tools.llm_tools as llm_tools
import src.agents.steward_agent as steward_agent
import src.agents.agent_runner as agent_runner

importlib.reload(llm_tools)
importlib.reload(steward_agent)
importlib.reload(agent_runner)

# Clear cached agents so the latest code is used
agent_runner._healing_agent = None
agent_runner._steward_agent = None

from src.agents.agent_runner import handle_incident, handle_incident_steward

if not latest_incident:
    raise RuntimeError("No incident was created. If the product is configured to only warn, you may not see an incident.")

INCIDENT_ID = latest_incident["id"]

healing_state = handle_incident(INCIDENT_ID, severity="high")
steward_state = handle_incident_steward(INCIDENT_ID, severity="high")

{
    "incident_id": INCIDENT_ID,
    "healing": {
        "status": healing_state.get("status") if healing_state else None,
        "action": healing_state.get("action") if healing_state else None,
        "recommendation": healing_state.get("recommendation") if healing_state else None,
    },
    "steward": {
        "status": steward_state.get("status") if steward_state else None,
        "remediation_steps": (steward_state.get("remediation_steps") if steward_state else None),
    },
}

2026-01-05 03:16:27 | HealingAgent | INFO | Loading incident INC_ecca6472


[AGENT] Healing Agent invoked for Incident INC_ecca6472
{"timestamp": "2026-01-04T21:46:27.615181+00:00", "level": "INFO", "logger": "dpos.agents.HealingAgent", "message": "Loading incident INC_ecca6472", "module": "healing_agent", "function": "load_incident", "line": 78}


2026-01-05 03:16:27 | HealingAgent | INFO | Analyzing impact for product DP001


{"timestamp": "2026-01-04T21:46:27.937768+00:00", "level": "INFO", "logger": "dpos.agents.HealingAgent", "message": "Analyzing impact for product DP001", "module": "healing_agent", "function": "analyze_impact", "line": 91}
{"timestamp": "2026-01-04T21:46:29.102755+00:00", "level": "INFO", "logger": "src.agents.tools.llm_tools", "message": "Root cause analysis completed for INC_ecca6472", "module": "logging", "function": "_log", "line": 134, "extra": {"confidence": "medium"}}


2026-01-05 03:16:29 | HealingAgent | INFO | Root cause analysis complete


...

2026-01-05 03:16:32 | StewardAgent | INFO | Generating standard remediation plan with LLM


{"timestamp": "2026-01-04T21:46:32.016213+00:00", "level": "INFO", "logger": "dpos.agents.StewardAgent", "message": "Generating standard remediation plan with LLM", "module": "steward_agent", "function": "standard_remediation", "line": 177}
{"timestamp": "2026-01-04T21:46:40.245635+00:00", "level": "INFO", "logger": "src.core.llm", "message": "LLM invocation successful", "module": "logging", "function": "_log", "line": 134, "extra": {"provider": "ollama", "latency_ms": 8222.418785095215, "cached": false}}
{"timestamp": "2026-01-04T21:46:50.019394+00:00", "level": "INFO", "logger": "src.core.llm", "message": "LLM invocation successful", "module": "logging", "function": "_log", "line": 134, "extra": {"provider": "ollama", "latency_ms": 9768.805980682373, "cached": false}}


2026-01-05 03:16:50 | StewardAgent | INFO | Notifying stakeholders with LLM-generated narrative


{"timestamp": "2026-01-04T21:46:50.038431+00:00", "level": "INFO", "logger": "dpos.agents.StewardAgent", "message": "Notifying stakeholders with LLM-generated narrative", "module": "steward_agent", "function": "notify_stakeholders", "line": 221}
{"timestamp": "2026-01-04T21:46:59.318059+00:00", "level": "INFO", "logger": "src.core.llm", "message": "LLM invocation successful", "module": "logging", "function": "_log", "line": 134, "extra": {"provider": "ollama", "latency_ms": 9272.715330123901, "cached": false}}
[AGENT] Steward Agent finished
  Status: completed
  Remediation Steps: 3


{'incident_id': 'INC_ecca6472',
 'healing': {'status': 'completed',
  'action': 'activate_fallback',
  'recommendation': 'Fallback source activated automatically. Root cause: A data processing error or pipeline failure introduced invalid, missing, or malformed data into the customer_profiles product, causing it to violate the defined data contract for product DP001.. Confidence: medium.'},
 'steward': {'status': 'completed',
  'remediation_steps': ['**Stop the Data Flow:** Immediately halt the data pipeline feeding the `customer_profiles` product (DP001) to prevent further corruption.',
   '**Assess Scope & Impact:** Identify the exact records, time range, and data fields affected by the processing error. Quantify the impact on downstream consumers.',
   '**Communicate with Stakeholders:** Notify all teams and services dependent on the `customer_profiles` product about the data quality issue, its severity, and the expected timeline for resolution.']}}

## 8) Impact (Who/What Is Affected)
If a product is degraded, DPOS can explain what downstream products/pipelines/users are impacted.

**Functional outcome:** you get a practical business impact summary and a risk score.

In [12]:
from src.agents.agent_runner import analyze_impact

impact_state = analyze_impact(PRODUCT_ID, incident_id=INCIDENT_ID)

{
    "risk_score": impact_state.get("risk_score") if impact_state else None,
    "business_impact": impact_state.get("business_impact") if impact_state else None,
    "downstream_products": (impact_state.get("downstream_products") if impact_state else None),
}

2026-01-05 03:16:59 | ImpactAgent | INFO | Loading product DP001


[IMPACT] Analysis for DP001
{"timestamp": "2026-01-04T21:46:59.843941+00:00", "level": "INFO", "logger": "dpos.agents.ImpactAgent", "message": "Loading product DP001", "module": "impact_agent", "function": "load_product", "line": 83}


2026-01-05 03:17:00 | ImpactAgent | INFO | Analyzing downstream dependencies


{"timestamp": "2026-01-04T21:47:00.224027+00:00", "level": "INFO", "logger": "dpos.agents.ImpactAgent", "message": "Analyzing downstream dependencies", "module": "impact_agent", "function": "analyze_downstream", "line": 96}


2026-01-05 03:17:01 | ImpactAgent | INFO | Calculating risk score with LLM assessment


...

2026-01-05 03:17:08 | ImpactAgent | INFO | Risk calculation complete


{"timestamp": "2026-01-04T21:47:08.195997+00:00", "level": "INFO", "logger": "dpos.agents.ImpactAgent", "message": "Risk calculation complete", "module": "impact_agent", "function": "calculate_risk", "line": 147}


2026-01-05 03:17:08 | ImpactAgent | INFO | Generating impact narrative with LLM


{"timestamp": "2026-01-04T21:47:08.199998+00:00", "level": "INFO", "logger": "dpos.agents.ImpactAgent", "message": "Generating impact narrative with LLM", "module": "impact_agent", "function": "generate_narrative", "line": 167}
{"timestamp": "2026-01-04T21:47:15.748536+00:00", "level": "INFO", "logger": "src.core.llm", "message": "LLM invocation successful", "module": "logging", "function": "_log", "line": 134, "extra": {"provider": "ollama", "latency_ms": 7546.537637710571, "cached": false}}
{"timestamp": "2026-01-04T21:47:22.169766+00:00", "level": "INFO", "logger": "src.core.llm", "message": "LLM invocation successful", "module": "logging", "function": "_log", "line": 134, "extra": {"provider": "ollama", "latency_ms": 6417.223691940308, "cached": false}}
[IMPACT] Analysis complete
  Risk Score: 9.0
  Business Impact: low
  Downstream Products: 0


{'risk_score': 9.0, 'business_impact': 'low', 'downstream_products': []}

## 9) Monitoring (SLA-Style Health Signals)
Monitoring highlights which products are at risk so you can prioritize attention.

**Functional outcome:** a short list of “at risk” and “breached” items.

In [13]:
try:
    from src.agents.sla_agent import run_sla_monitoring

    sla_report = run_sla_monitoring([PRODUCT_ID])
    {
        "status": sla_report.get("status"),
        "breached_slas": sla_report.get("breached_slas", [])[:10],
        "at_risk_slas": sla_report.get("at_risk_slas", [])[:10],
    }
except Exception as e:
    {"status": "not_available", "reason": str(e)}

2026-01-05 03:17:22 | SLAAgent | INFO | Loading SLAs and metrics


{"timestamp": "2026-01-04T21:47:22.287252+00:00", "level": "INFO", "logger": "dpos.agents.SLAAgent", "message": "Loading SLAs and metrics", "module": "sla_agent", "function": "load_slas", "line": 241}


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: target_value)} {position: line: 5, column: 22, offset: 200} for query: '\n            MATCH (p:DataProduct)-[:HAS_CONTRACT]->(c:Contract)-[:HAS_SLA]->(s:SLA)\n            WHERE p.id IN $product_ids\n            RETURN s.id as sla_id, s.name as sla_name,\n                   s.target_value as target_value, s.metric_type as metric_type,\n                   s.threshold as threshold, s.is_active as is_active,\n                   c.id as contract_id, c.name as contract_name,\n                   p.id as product_id, p.name as product_name\n            '
2026-01

{"timestamp": "2026-01-04T21:47:22.691363+00:00", "level": "INFO", "logger": "dpos.agents.SLAAgent", "message": "Loaded 0 SLAs for 0 products", "module": "sla_agent", "function": "load_slas", "line": 253}


2026-01-05 03:17:22 | SLAAgent | INFO | Checking SLA compliance


...

{"timestamp": "2026-01-04T21:47:22.750376+00:00", "level": "INFO", "logger": "dpos.agents.SLAAgent", "message": "Identified 0 SLAs at risk", "module": "sla_agent", "function": "predict_risks", "line": 328}


2026-01-05 03:17:22 | SLAAgent | INFO | Generating SLA compliance report


{"timestamp": "2026-01-04T21:47:22.756408+00:00", "level": "INFO", "logger": "dpos.agents.SLAAgent", "message": "Generating SLA compliance report", "module": "sla_agent", "function": "generate_report", "line": 422}
{"timestamp": "2026-01-04T21:47:28.003153+00:00", "level": "INFO", "logger": "src.core.llm", "message": "LLM invocation successful", "module": "logging", "function": "_log", "line": 134, "extra": {"provider": "ollama", "latency_ms": 5244.7829246521, "cached": false}}


2026-01-05 03:17:28 | SLAAgent | INFO | Persisting SLA monitoring results


{"timestamp": "2026-01-04T21:47:28.008179+00:00", "level": "INFO", "logger": "dpos.agents.SLAAgent", "message": "Persisting SLA monitoring results", "module": "sla_agent", "function": "persist_results", "line": 476}


## 10) Streaming Concept (Real-Time Gatekeeping)
DPOS can apply the same “validate → enforce” logic to streaming events.

**Functional outcome:** a “good event” routes forward; a “bad event” gets blocked or diverted.

In [16]:
from src.utils.config import app_config
from src.enforcement.streaming_enforcer import StreamingEnforcer

enforcer = StreamingEnforcer()
raw_topic = f"{app_config.topic_prefix_raw}{PRODUCT_ID}"

# “Good” outreach event: contactable record
good_msg = ("{" + "\"customer_id\":\"123\",\"email\":\"test@test.com\",\"name\":\"Alice\",\"segment\":\"premium\"}" ).encode("utf-8")

# “Bad” outreach event: missing contact + invalid segment
bad_msg = ("{" + "\"customer_id\":\"123\",\"email\":null,\"name\":\"\",\"segment\":\"bogus\"}" ).encode("utf-8")

{
    "topic": raw_topic,
    "good_result": enforcer.process_message(raw_topic, good_msg),
    "bad_result": enforcer.process_message(raw_topic, bad_msg),
}

[#C8E0]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('si-84dce434-6c33.production-orch-0695.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.161.242', 7687))): OSError('No data')
Transaction failed and will be retried in 1.085290586033143s (Failed to read from defunct connection IPv4Address(('si-84dce434-6c33.production-orch-0695.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.161.242', 7687))))
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownRelationshipTypeWarning} {category: UNRECOGNIZED} {title: The provided relationship type is not in the database.} {description: One of the relationship types in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing relationship type is: APPLIES_TO_PRODUCT)} {position: line: 3, column: 18, offset: 77} for query: "\n        MATCH (p:Policy {scope:'

{'topic': 'dpos.raw.DP001',
 'good_result': {'product_id': 'DP001',
  'status': 'failed',
  'action': 'blocked'},
 'bad_result': {'product_id': 'DP001',
  'status': 'failed',
  'action': 'blocked'}}

## 11) Supervisor (End-to-End Summary + Next Actions)
The supervisor produces an executive-friendly answer by coordinating multiple capabilities.

**Functional outcome:** one coherent narrative: what happened, what it means, what to do next.

In [15]:
try:
    import uuid
    import importlib

    # Reload agents in case code changed while the kernel is running
    import src.agents.insights_agent as insights_agent
    import src.agents.supervisor_agent as supervisor_agent

    importlib.reload(insights_agent)
    importlib.reload(supervisor_agent)

    from src.agents.supervisor_agent import build_supervisor_agent
    from src.agents.checkpointer import get_thread_config

    supervisor = build_supervisor_agent()
    q = (
        f"Dataset quality incident (usecase={USECASE}): summarize what failed, "
        "who is impacted, and propose a prioritized action plan. "
        f"Context: product={PRODUCT_ID}, incident={INCIDENT_ID}."
    )
    result = supervisor.invoke(
        {
            "query": q,
            "conversation_history": [],
            "intent": None,
            "intent_confidence": None,
            "execution_plan": [],
            "current_step": 0,
            "agent_results": {},
            "intermediate_findings": [],
            "requires_human_approval": False,
            "approval_reason": None,
            "approved": False,
            "response": None,
            "recommendations": [],
            "follow_up_questions": [],
            "execution_log": [],
            "total_agents_invoked": 0,
            "status": "new",
            "messages": [],
        },
        config=get_thread_config(f"walkthrough_supervisor_{uuid.uuid4().hex[:8]}"),
    )

    {
        "intent": result.get("intent"),
        "agents_invoked": result.get("total_agents_invoked"),
        "response": result.get("response"),
        "recommendations": result.get("recommendations", [])[:8],
        "follow_up_questions": result.get("follow_up_questions", [])[:8],
    }
except Exception as e:
    {"status": "not_available", "reason": str(e)}

2026-01-05 03:17:32 | SupervisorAgent | INFO | Understanding intent for query: Dataset quality incident (usecase=ecommerce): summarize what failed, who is impacted, and propose a ...


{"timestamp": "2026-01-04T21:47:32.402821+00:00", "level": "INFO", "logger": "dpos.agents.SupervisorAgent", "message": "Understanding intent for query: Dataset quality incident (usecase=ecommerce): summarize what failed, who is impacted, and propose a ...", "module": "supervisor_agent", "function": "understand_intent", "line": 438}
{"timestamp": "2026-01-04T21:47:38.480343+00:00", "level": "INFO", "logger": "src.core.llm", "message": "LLM invocation successful", "module": "logging", "function": "_log", "line": 134, "extra": {"provider": "ollama", "latency_ms": 6076.329946517944, "cached": false}}


2026-01-05 03:17:38 | SupervisorAgent | INFO | Classified intent: INCIDENT_HANDLING (confidence: 1.0)


{"timestamp": "2026-01-04T21:47:38.483137+00:00", "level": "INFO", "logger": "dpos.agents.SupervisorAgent", "message": "Classified intent: INCIDENT_HANDLING (confidence: 1.0)", "module": "supervisor_agent", "function": "understand_intent", "line": 493}


2026-01-05 03:17:38 | SupervisorAgent | INFO | Creating execution plan


...

{"timestamp": "2026-01-04T21:47:48.982011+00:00", "level": "INFO", "logger": "dpos.agents.InsightsAgent", "message": "Persisting insights", "module": "insights_agent", "function": "persist_insights", "line": 586}


2026-01-05 03:17:49 | SupervisorAgent | INFO | Synthesizing response


{"timestamp": "2026-01-04T21:47:49.366161+00:00", "level": "INFO", "logger": "dpos.agents.SupervisorAgent", "message": "Synthesizing response", "module": "supervisor_agent", "function": "synthesize_response", "line": 707}
{"timestamp": "2026-01-04T21:47:57.456896+00:00", "level": "INFO", "logger": "src.core.llm", "message": "LLM invocation successful", "module": "logging", "function": "_log", "line": 134, "extra": {"provider": "ollama", "latency_ms": 8088.884115219116, "cached": false}}


2026-01-05 03:17:57 | SupervisorAgent | INFO | Persisting supervisor execution


{"timestamp": "2026-01-04T21:47:57.462046+00:00", "level": "INFO", "logger": "dpos.agents.SupervisorAgent", "message": "Persisting supervisor execution", "module": "supervisor_agent", "function": "persist_execution", "line": 818}


## 12) Claude Desktop Demo (Functional Prompts)
In Claude Desktop, you can use the DPOS MCP tools to do the same things, but conversationally.

### How to confirm which agent ran
- In Claude Desktop you should see a tool-call UI (e.g. calling `dpos_handle_incident`). That tells you which MCP tool was used.
- DPOS MCP tools also return a `_meta` block (e.g. `_meta.agent: HealingAgent`) so you can verify the agent behind the tool.
- If available, tools also return `_meta.llm` so you can see whether an LLM was used (`used=true/false`) and which provider/model handled it.
- If Claude answers without tool calls, then **no DPOS agent/tool actually ran** — it was just a conversational answer.
  - If you do **not** see `_meta.llm` (or you keep seeing old errors), Claude Desktop is likely using an older MCP server process. Fully restart Claude Desktop / the MCP server.

### Troubleshooting (common)
- If `dpos_monitor_slas` fails with: `Checkpointer requires ... thread_id ...`, your MCP server is running an older build OR is using a Python environment missing the repo dependencies.
  - Ensure Claude Desktop launches MCP with the repo venv Python and the repo entrypoint (`dpos-ecommerce/run_dpos_mcp.py`).
  - Install dependencies in that venv (repo root): `pip install -r requirements.txt` (includes `pydantic`).
  - Restart Claude Desktop and retry.

### Copy/paste test script (runs end-to-end, forces tool usage)
Paste these one-by-one in order. This sequence is designed to exercise discovery, validation evidence, incident handling, impact, insights, and the supervisor orchestration.

0) Ecommerce dataset (good vs bad batches)
Use this when you ran the walkthrough with `DPOS_USECASE=ecommerce` (it uses: `data/usecase/ecommerce/good/customers.csv` and `data/usecase/ecommerce/bad/customers_bad.csv`).
- "Use DPOS tools only. Call `dpos_execute_cypher` to fetch the latest 2 `ValidationReport` records for product_id='DP001'. Return: report id, result/action, pass/fail counts, and timestamp."
  - Cypher:
    - `MATCH (p:DataProduct {id:'DP001'})-[:HAS_VALIDATION]->(v:ValidationReport)`
    - `RETURN v.id AS id, v.result AS result, v.action AS action, v.total_records AS total, v.passed_records AS passed, v.failed_records AS failed, coalesce(v.created_at, v.timestamp) AS ts`
    - `ORDER BY ts DESC LIMIT 2`
- "Use DPOS tools only. For each of those report ids, call `dpos_execute_cypher` to return `v.violations` and summarize the top 5 issues (treat it as JSON)."
  - Cypher: `MATCH (v:ValidationReport {id:$id}) RETURN v.violations AS violations, v.result AS result, v.action AS action`
- "Use DPOS tools only. Explain the business impact difference between the ecommerce 'good' batch (customers.csv) vs the ecommerce 'bad' batch (customers_bad.csv) based on the violations."
- "Use DPOS tools only. If an incident exists for DP001, call `dpos_list_incidents` (status='open') and return the newest incident id for DP001 so I can run Healing/Steward tools next."
  - If you don’t see ecommerce validation reports yet: run the notebook once with `DPOS_USECASE=ecommerce` to generate them.

1) Catalog discovery (force tools)
- "Use DPOS tools only. Call `dpos_get_dashboard_stats` and summarize the current state."
- "Use DPOS tools only. Call `dpos_search_products` with query='customer' and show the top results."

2) Contract expectations (what “good” means)
- "Use DPOS tools only. Call `dpos_list_contracts` and identify the contract linked to product DP001 (if any)."
- "Then call `dpos_get_contract_rules` for that contract and summarize the key expectations in plain language."

3) Validation evidence (what failed)
- "Use DPOS tools only. Call `dpos_execute_cypher` to find the latest `ValidationReport` for product DP001 and summarize pass/fail counts and top violations."

4) Find the latest incident id for DP001 (needed to run Healing/Steward tools)
- "Use DPOS tools only. Call `dpos_list_incidents` with status='open' and find the most recent incident_id for product_id='DP001'. Return just the incident_id."

5) Incident handling (HealingAgent)
- "Use DPOS tools only. Call `dpos_handle_incident` with incident_id=<the id from step 4> and severity='high'. Return the action + recommendation."

6) Incident handling (StewardAgent)
- "Use DPOS tools only. Call `dpos_steward_review` with the same incident_id and severity='high'. Return the governance recommendations."

7) Impact analysis (ImpactAgent)
- "Use DPOS tools only. Call `dpos_analyze_impact` for product_id='DP001'."

8) Monitoring / SLA signals (SLAAgent)
- "Use DPOS tools only. Call `dpos_monitor_slas` for product_ids=['DP001'] and summarize any breaches/risks."

9) Insights (InsightsAgent)
- "Use DPOS tools only. Call `dpos_generate_insights` for time_range_days=30 and summarize the key findings."

10) Supervisor orchestration (SupervisorAgent)
- "Use DPOS tools only. Call `dpos_supervisor_query` with: 'Summarize what failed in the latest DP001 incident, who is impacted, and propose a prioritized action plan. Show evidence.'"

---
### Business-first prompts (Claude may or may not choose tools unless you insist)
### A. Discover products for your domain (Catalog / Discovery)
- "Show me the data products relevant to my domain (e.g., customer analytics or public reporting). Summarize what each product provides and who owns it."
- "Which products are currently degraded or have open incidents? Summarize the business risk."
- "List products that do NOT have an active contract yet, and recommend which ones need contracts first."

---
### B. Validate data quality (Validation / Contract outcomes)
- "Show the last 5 validation reports. For each, summarize: product, result, action, and top 3 failure reasons."
- "For the latest failed validation report for DP001, explain what the bad rows have in common and what to fix first."
- "If this kind of failure repeats weekly, what contract rules should we tighten or add?"

#### Ecommerce-specific (customers.csv vs customers_bad.csv)
- "Assume USECASE=ecommerce. Compare the latest PASS vs FAIL validation reports for DP001. What changed between the good and bad batches, and which 3 contract checks are most valuable for ecommerce customer/contact data?"
- "For ecommerce, translate the top violations into downstream impact: CRM segmentation, email deliverability, duplicate outreach, and compliance risk."
- "If we had to ship a temporary workaround for the bad ecommerce batch, propose the safest mitigation (quarantine vs partial publish) and what fields/rows you would exclude."

---
### C. Enforcement decisions (Publish vs Block vs Quarantine)
- "For DP001, explain what action DPOS took on the last validation and why (publish/block/quarantine)."
- "If enforcement is currently set to 'warn', what risks does that create and what should we change?"
- "Show me any quarantined batches and what is needed to release them safely."

---
### D. Incidents (Create / Triage / Explain)
- "List the most recent high-severity incidents and explain what went wrong in business terms."
- "For the latest incident: what is the likely root cause, and what evidence supports it?"
- "What are the top 3 recurring incident patterns in the last 30 days?"

---
### E. Healing agent (Operations mitigation)
- "If we cannot fix upstream immediately, propose a safe temporary workaround so downstream users can keep operating."
- "Draft a short message I can send to stakeholders: what happened and what we’re doing now."

---
### F. Steward agent (Governance remediation)
- "What changes should we make to contracts/policies to prevent this class of incident?"
- "Propose a stewardship follow-up plan with owners and due dates (high level)."

---
### G. Impact agent (Downstream effects / risk)
- "If DP001 is degraded, which downstream products/pipelines/teams are affected? Summarize impact and provide a prioritized action plan."
- "What is the estimated business risk if this lasts 1 hour vs 24 hours?"

---
### H. SLA / Monitoring agent (Health signals)
- "What monitoring signals would have detected this earlier? Propose 3 new alerts."

---
### I. Insights agent (Trends + patterns)
- "Which products are the riskiest right now and why? Provide a short prioritized list."

---
### J. Supervisor tool (explicitly test orchestration)
- "Use the supervisor tool to answer: summarize what failed in the latest DP001 incident, who is impacted, and propose a prioritized action plan. Show the evidence you used."
- "Use the supervisor tool to produce an executive incident brief for leadership."
- "Use the supervisor tool to propose follow-up questions and a next-step checklist."

---
### If you want Claude to drive via tools explicitly
- "Use the available DPOS tools to answer, and show the evidence you used."